# Project 12 — Errors-in-Variables (Measurement-Error Model)

**Scenario.** We want the relationship $y = \alpha + \beta x^*$, but the instrument adds noise to **both** variables. We never see the true predictor $x^*$; we see $x_\text{obs} = x^* + \text{noise}$ and a noisy $y$. Regressing $y$ on $x_\text{obs}$ — the obvious thing — **attenuates** the slope toward 0.

**New skill:** model noise in a *predictor* via a **latent true predictor** $x^*$. **Key pitfall:** attenuation bias if predictor noise is ignored.

In [ ]:
import sys, pathlib
sys.path.insert(0, r'/home/user/biofx_python/bayesian_workflow_portfolio')
sys.path.insert(0, str(pathlib.Path.cwd()))
import warnings; warnings.filterwarnings('ignore')

In [ ]:
import numpy as np
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt
az.style.use('arviz-darkgrid')
RNG = 20240601

## Step 1 — Problem & data-generating story

$$x^*_i \sim \text{Normal}(\mu_x, s_x), \quad x_{\text{obs},i} = x^*_i + \text{Normal}(0, \tau_x), \quad y_i = \alpha + \beta x^*_i + \text{Normal}(0, \sigma_y).$$

**Assumptions:** (a) the structural relationship is linear in the *true* $x^*$, (b) predictor noise is additive Gaussian with known SD $\tau_x$ (from calibration), (c) the response noise is independent of the predictor noise. Truth: $\alpha=1,\ \beta=2,\ \sigma_y=0.5,\ \tau_x=0.6$. The **attenuation factor** $s_x^2/(s_x^2+\tau_x^2)\approx0.74$ predicts how far the naive slope is pulled toward 0.

In [ ]:
from data.generate_data import generate
data = generate()
x_obs, y = data['x_obs'], data['y']
print(f"n={data['n']}, true beta={data['truth']['beta']}, "
      f"tau_x={data['tau_x']}, attenuation factor={data['attenuation_factor']:.2f}")
b_ols = np.cov(x_obs, y)[0,1]/np.var(x_obs)
print(f'naive OLS slope (y on x_obs) = {b_ols:.3f}  (attenuated!)')

## Step 2 — Two models

**Naive:** $y_i \sim \text{Normal}(\alpha + \beta x_{\text{obs},i}, \sigma_y)$ — regress on the noisy predictor (biased).

**Errors-in-variables (EIV):** treat $x^*$ as a **latent** variable with a population prior, add a measurement model $x_\text{obs} \sim \text{Normal}(x^*, \tau_x)$, and write the structural model in terms of $x^*$. With $\tau_x$ known, the slope is de-attenuated.

Priors: $\alpha,\beta \sim \text{Normal}(0,5)$ on the regression coefficients; $\mu_x \sim \text{Normal}(0,5)$, $s_x \sim \text{HalfNormal}(5)$ for the latent-predictor population.

In [ ]:
from model import build_model, fit
build_model(data, model='eiv')

## Step 3 — Prior predictive checks

We confirm the priors imply plausible $y$ ranges before fitting.

In [ ]:
with build_model(data, model='eiv') as m:
    prior = pm.sample_prior_predictive(draws=300, random_seed=RNG)
pp = prior.prior_predictive['y'].values.ravel()
fig, ax = plt.subplots(figsize=(6,3.5))
ax.hist(pp, bins=40, color='#55A868', edgecolor='white', density=True)
ax.axvline(y.mean(), color='red', lw=1.5, label='observed mean y')
ax.set(xlabel='y implied by prior', ylabel='density', title='Prior predictive')
ax.legend(); plt.tight_layout()

## Step 4 — Inference (both models)

We fit the naive model and the EIV model. The EIV model has 100s of latent $x^*_i$, so we use `target_accept=0.95` and ample tuning.

In [ ]:
idata_naive = fit(data, model='naive', draws=800, tune=1000, chains=4, seed=101)
idata_eiv = fit(data, model='eiv', draws=800, tune=1500, chains=4,
                target_accept=0.95, seed=101)

## Step 5 — Diagnostics

Check $\hat R$, ESS, and divergences for both. The EIV model is harder to sample (a high-dimensional latent vector); `sigma_y` is the most weakly identified parameter (it trades off against the predictor-noise scale), so expect its ESS to be the lowest — the **slope** $\beta$, our target, is well-behaved.

In [ ]:
print('NAIVE'); print(az.summary(idata_naive, var_names=['alpha','beta','sigma_y']))
print('\nEIV'); print(az.summary(idata_eiv, var_names=['alpha','beta','sigma_y']))
print('EIV divergences:', int(idata_eiv.sample_stats['diverging'].sum()))

## Step 6 — Posterior predictive checks

Both models can fit the *observed* $y$ adequately — a reminder that a good PPC on $y$ does **not** vindicate the naive model's *slope*. The bias is in the coefficient, not necessarily the fit to $y$.

In [ ]:
az.plot_ppc(idata_eiv, num_pp_samples=100); plt.tight_layout()

## Step 7 — The headline: naive vs EIV slope

Overlay the two posterior slopes against the true $\beta=2$. The naive posterior sits near the attenuated value ($\approx \beta \times 0.74$); the EIV posterior covers the truth.

In [ ]:
fig, ax = plt.subplots(figsize=(6.5,4))
az.plot_dist(idata_naive.posterior['beta'].values.ravel(), ax=ax,
             color='#C44E52', label='naive (attenuated)')
az.plot_dist(idata_eiv.posterior['beta'].values.ravel(), ax=ax,
             color='#4C72B0', label='errors-in-variables')
ax.axvline(data['truth']['beta'], color='k', ls='--', label='true beta=2')
ax.set(xlabel='slope beta', ylabel='density', title='Attenuation and its correction')
ax.legend(); plt.tight_layout()

## Step 8 — Decision & communication

Recover $(\alpha,\beta)$ of the EIV model and verify against truth.

In [ ]:
from shared.bayes_utils import check_recovery
truths = {k: data['truth'][k] for k in ['alpha','beta']}
for res in check_recovery(idata_eiv, truths):
    print(res)

**Conclusion (for a collaborator).** The true effect of $x$ on $y$ is $\beta\approx2$, but the naive regression reports only ~1.5 because the predictor is measured with noise. Quoting the naive slope would understate the effect by ~25%. The fix requires knowing (calibrating) the predictor's measurement-error SD $\tau_x$. See `summary_onepager.md`.